## Important: 
The new solution (MAD framework) currently lacks support of some legacy functionality, which is critical for identical behaviour between solutions. To ensure continuity, the legacy solution was configured to use the same convention and methods as the new solution. This approach allows us to address immediate needs, such as deriving comparable legacy values, while providing time to enhance the new solution to include the missing features.

To derive the legacy values, the following adaptation to the default settings of the legacy solution was applied: 

Configured interpolator type for runner:

```python
  interpolator=MergeSortInterpolator(
      default_method=SciPyInterpolation.PREVIOUS
  )
```

Adapt the calculation of for differences in the signal where 
data is a pandas DataFrame

```python
  data = data.diff()
  data.values[0] = 0
  return data                         
```

This is done to reproduce the same behaviour as the MDA framework. 
```python
  data.values[0] = 0
  data.values[1:] = data.values[1:] - data.values[:-1]
```

This setup was run on the **Measurement** Event, as definied in the FleetMonitioringTB with the follwoing parameters set in the appropriate .yaml file:

```yaml
histograms:
  enable: true
  configs:
    - discretization_channel: SpdEng
      aggr_channel: time
      bin_size: 250
      aggr: [sum]
    - discretization_channel: SpdVeh
      aggr_channel: time
      bin_size: 5
      aggr: [sum]
    - discretization_channel: TAmb
      aggr_channel: time
      bin_size: 5
      aggr: [sum]
      factor: 0.001
    - discretization_channel: SpdVeh
      aggr_channel: dummy
      bin_size: 50
      aggr: [sum]
    - discretization_channel: SpdVeh
      aggr_channel: OdoVeh
      bin_size: 10
      aggr: [sum]
heatmaps:
  enable: true
  configs:
    - discretization_channel_x: SpdEng
      discretization_channel_y: SpdVeh
      aggr_channel: OdoVeh
      bin_size_x: 500
      bin_size_y: 25
    - discretization_channel_x: SpdEng
      discretization_channel_y: TqEng
      aggr_channel: time
      bin_size_x: 200
      bin_size_y: 200
calculated_channels:
  dummy:
   # this formual mocks up a time channel under a different name ('time' is specially treated),
   # allowing us to have a constant weight, while still beeing custom.
    formula: "SpdEng*0 + time" 
    required_channels:
      - SpdEng 

```

Additionally the followoing channelmapping was used:
```yaml
  tenant-url: https://dap-teamtesting-toolbox.atgrzck4022.atgrz.onprem.avl.zone
  project-id: 29
  channel-mapping-application-code: databricks
  channel-mapping-version-number: 9
```

### General Setup

In [0]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pyspark.sql.functions as F
from pytest import approx
from mda_query_engine.analyze.metadata.tag_expression import *
from mda_query_engine.analyze.query.solvers.basic_narrow_solver import BasicNarrowSolver
from mda_query_engine.measurement_db import MeasurementDB, MeasurementDBConfig
from mda_query_engine.model.series.sample_series import SampleSeries
from mda_reporting.aggregations.histogram import (
    HistogramDuration,
    HistogramCustomWeights,
    HistogramDistance,
)
from mda_reporting.aggregations.histogram2d import (
    Histogram2DDuration,
    Histogram2DCustomWeights,
    Histogram2DDistance,
)
from mda_reporting.core.page import Page
from mda_reporting.core.report import Report
from mda_reporting.events.basic_event import BasicEvent

### Initialization

In [0]:
dbutils.widgets.text("catalog_in", "avl_databricks_mvp")
dbutils.widgets.text("schema_in", "silver_zstd")
dbutils.widgets.text("catalog_out", "development")
dbutils.widgets.text("schema_out", "gold_e2e")
dbutils.widgets.text("prefix_out", "legacy")
dbutils.widgets.text("reset", "false")

catalog_in = dbutils.widgets.get("catalog_in")
schema_in = dbutils.widgets.get("schema_in")
catalog_out = dbutils.widgets.get("catalog_out")
schema_out = dbutils.widgets.get("schema_out")
prefix_out = dbutils.widgets.get("prefix_out")
reset = dbutils.widgets.get("reset")

# gold_target:

if reset == "true":
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_histogram_fact"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_histogram2d_fact"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_histogram_dimension"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_histogram2d_dimension"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_event_instance_fact"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_event_dimension"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_measurement_dimension"
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_out}.{schema_out}")

In [0]:
config = json.load(open("./config/config.json"))
config["source"][
    "container_metrics_table"
] = f"{catalog_in}.{schema_in}.container_metric"
config["source"]["channel_metrics_table"] = f"{catalog_in}.{schema_in}.channel_metric"
config["source"]["channels_uri"] = f"{catalog_in}.{schema_in}.channel_data"

config["unity_sink"]["catalog"] = catalog_out
config["unity_sink"]["schema"] = schema_out
config["unity_sink"]["table_prefix"] = prefix_out

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")
my_report = Report(
    name="my_report", spark=spark, config=config
)  # config_path="./config/config.json")
db = my_report.get_db()

### Signal Queries & Virtual Signal Creation

In [0]:
# physical signals
eng_rpm = db.query.channel(channel_name="is1_eng_speed", data_key="TM")
veh_spd = db.query.channel(channel_name="can_vehicle_speed", data_key="TM")
trq_current = db.query.channel(channel_name="isp_trq_current", data_key="TM")
t_ambient_air = db.query.channel(channel_name="isp_t_ambient_air", data_key="TM")
p_ambient_air = db.query.channel(channel_name="is4_p_ambient_air", data_key="TM")
odo_km = db.query.channel(
    channel_name="can_in_CGW_C1_TotalVehDist_Cval_DIAG", data_key="TM"
)

# virtual signals
# power = eng_rpm * trq_current / 5252.0
dummy_signal = (veh_spd * 0) + 1
# event expressions
veh_spd_event = veh_spd > -1
eng_rpm_event = eng_rpm > -1
speed_above_0_event = veh_spd > -1
odo_above_0_event = odo_km > -1

### Event Definitions
All Events are defined in a way where all datapoints return _true_, effectivly leading to a '_Measurement_' event.

In [0]:
veh_spd_event = BasicEvent(
    name="speed_event",
    expr=veh_spd_event,
    desc="Vehicle speed > -1",
    required_channels=["can_vehicle_speed"],
)
rpm_event = BasicEvent(
    name="eng_rpm_event",
    expr=eng_rpm_event,
    desc="Engine RPM > -1",
    required_channels=["is1_eng_speed"],
)
speed_above_0_event = BasicEvent(
    name="speed_above_0_event",
    expr=speed_above_0_event,
    desc="Vehicle speed > -1",
    required_channels=["can_vehicle_speed"],
)
odo_above_0_event = BasicEvent(
    name="odo_above_0_event",
    expr=odo_above_0_event,
    desc="odo km > -1",
    required_channels=["can_in_CGW_C1_TotalVehDist_Cval_DIAG"],
)
my_report.add_event(veh_spd_event)
my_report.add_event(rpm_event)
my_report.add_event(speed_above_0_event)
my_report.add_event(odo_above_0_event)

### Histogram Definition

The ranges for the histograms where set to reproduce the same bin values as the legacy solution which uses automatic bin definition via bin size alone.

In [0]:
# Definition of 1st page
my_first_page = Page(page_number=1)
my_report.add_page(my_first_page)

# RPM histogram within speed events
hist1_name = "rpm_hist_p1"
hist1_desc = "Engine RPM histogram within speed events"
hist1_bins = [float(i) for i in range(0, 2750, 250)]
hist1 = HistogramDuration(
    name=hist1_name,
    base_expr=eng_rpm,
    event=rpm_event,
    bins=hist1_bins,
    desc=hist1_desc,
    channel_name="is1_eng_speed",
)
my_first_page.add_aggregation(hist1)

# Speed histogram within rpm events
hist2_name = "speed_hist_p1"
hist2_desc = "Vehicle speed histogram within RPM events"
hist2_bins = [float(i) for i in range(0, 100, 5)]
hist2 = HistogramDuration(
    name=hist2_name,
    base_expr=veh_spd,
    event=rpm_event,
    bins=hist2_bins,
    desc=hist2_desc,
    channel_name="can_vehicle_speed",
)
my_first_page.add_aggregation(hist2)

# Ambient Temperature histogram within rpm events
hist3_name = "t_amb_air_hist_p1"
hist3_desc = "Ambient temperature histogram within RPM events"
hist3_bins = [float(i) for i in range(0, 35, 5)]
hist3 = HistogramDuration(
    name=hist3_name,
    base_expr=t_ambient_air,
    event=rpm_event,
    bins=hist3_bins,
    desc=hist3_desc,
    channel_name="isp_t_ambient_air",
)
my_first_page.add_aggregation(hist3)

hist_5_name = "hist_custom_weights"
hist_5_desc = "histogram with custom weights"
hist5_bins = [float(i) for i in range(0, 150, 50)]
hist_5 = HistogramCustomWeights(
    name=hist_5_name,
    base_expr=veh_spd,
    weights_expr=dummy_signal,
    bins=hist5_bins,
    desc=hist_5_desc,
    channel_name="can_vehicle_speed",
    weight_type="time",
)
my_first_page.add_aggregation(hist_5)

hist_6_name = "histogram_1d_distance"
hist_6_desc = "histogram 1d distance values"
hist_6_bins = [float(i) for i in range(0, 110, 10)]
hist_6 = HistogramDistance(
    name=hist_6_name,
    base_expr=veh_spd,
    event=odo_above_0_event,  # speed_above_0_event,
    weights_expr=odo_km,
    bins=hist_6_bins,
    desc=hist_6_desc,
    channel_name="can_in_CGW_C1_TotalVehDist_Cval_DIAG",
)
my_first_page.add_aggregation(hist_6)

# Heatmap Engine Speed vs. torque
hist_4_name = "rpm_torque_heatmap"
hist_4_desc = "Engine RPM vs. torque heatmap within RPM events"
hist_4_x_bins = [float(i) for i in range(0, 2600, 200)]
hist_4_y_bins = [float(i) for i in range(-1400, 2400, 200)]
hist_4 = Histogram2DDuration(
    name=hist_4_name,
    x_expr=eng_rpm,
    y_expr=trq_current,
    event=rpm_event,
    x_bins=hist_4_x_bins,
    y_bins=hist_4_y_bins,
    desc=hist_4_desc,
    x_channel_name="is1_eng_speed",
    y_channel_name="isp_trq_current",
)
my_first_page.add_aggregation(hist_4)

histogram_7_name = "histogram_2d_custom_weights"
histogram_7_desc = "histogram 2d custom weights"
histogram_7_x_bins = [float(i) for i in range(0, 3000, 500)]
histogram_7_y_bins = [float(i) for i in range(0, 125, 25)]
histogram_7 = Histogram2DCustomWeights(
    name=histogram_7_name,
    x_expr=eng_rpm,
    y_expr=veh_spd,
    event=odo_above_0_event,
    weights_expr=odo_km,
    x_bins=histogram_7_x_bins,
    y_bins=histogram_7_y_bins,
    desc=histogram_7_desc,
    x_channel_name="is1_eng_speed",
    y_channel_name="can_vehicle_speed",
    weights_channel_name="can_in_CGW_C1_TotalVehDist_Cval_DIAG",
    math_fct_for_weights="diff",
    # weight_type='time'
)
my_first_page.add_aggregation(histogram_7)

histogram_8_name = "histogram_2d_distance"
histogram_8_desc = "histogram 2d distance"
histogram_8_x_bins = [float(i) for i in range(0, 3000, 500)]
histogram_8_y_bins = [float(i) for i in range(0, 125, 25)]
histogram_8 = Histogram2DDistance(
    name=histogram_8_name,
    x_expr=eng_rpm,
    y_expr=veh_spd,
    event=odo_above_0_event,
    weights_expr=odo_km,
    x_bins=histogram_8_x_bins,
    y_bins=histogram_8_y_bins,
    desc=histogram_8_desc,
    x_channel_name="is1_eng_speed",
    y_channel_name="can_vehicle_speed",
)
my_first_page.add_aggregation(histogram_8)

### Run Calculation and Exract Results from Gold Layer

In [0]:
my_report.determine_report()
my_report.persist_results()


hist_df = spark.read.table(f"{catalog_out}.{schema_out}.{prefix_out}_histogram_fact")
hist2d_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_histogram2d_fact"
)

hist_meta_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_histogram_dimension"
)
hist2d_meta_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_histogram2d_dimension"
)

event_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_event_instance_fact"
)
event_meta_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_event_dimension"
)
measurement_dim_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_measurement_dimension"
)

#### RPM Histogram

In [0]:
interesting_files = [
    "AA080518_Continuous_20250427_052054_20250427_100541.MF4",
]
result_histogram_rpm_duration = (
    hist_df.join(hist_meta_df, on="visual_id", how="inner")
    .where(F.col("name") == F.lit("rpm_hist_p1"))
    .join(
        F.broadcast(
            measurement_dim_df.where(F.col("file_name").isin(interesting_files))
        ),
        on="container_id",
        how="inner",
    )
    .groupBy(
        F.col("name"),
        F.col("bin_ID"),
        F.col("lower_bound"),
        F.col("upper_bound"),
        F.col("bin_name"),
    )
    .agg(F.sum(F.col("hist_value")).alias("hist_value"))
    .orderBy(F.col("bin_id").asc())
    .collect()
)

actual_values_histogram_rpm_duration = [
    row["hist_value"] for row in result_histogram_rpm_duration
]

# --------- Legacy Values ---------

legacy_values_histogram_rpm_duration = [
    x * 1e-3
    for x in [
        33200.0,
        375200.0000000461,
        648699.9999999446,
        196600.00000000122,
        1799100.000000026,
        8037900.000000025,
        3709299.999999943,
        1976300.000000016,
        292199.9999999922,
        18900.00000000547,
    ]
]

#### Speed Histogram

In [0]:
result_histogram_speed_duration = (
    hist_df.join(hist_meta_df, on="visual_id", how="inner")
    .where(F.col("name") == F.lit("speed_hist_p1"))
    .join(
        F.broadcast(
            measurement_dim_df.where(F.col("file_name").isin(interesting_files))
        ),
        on="container_id",
        how="inner",
    )
    .groupBy(
        F.col("name"),
        F.col("bin_ID"),
        F.col("lower_bound"),
        F.col("upper_bound"),
        F.col("bin_name"),
    )
    .agg(F.sum(F.col("hist_value")).alias("hist_value"))
    .orderBy(F.col("bin_id").asc())
    .select("hist_value")
    .collect()
)

actual_values_histogram_speed_duration = [
    row["hist_value"] for row in result_histogram_speed_duration
]

# --------- Legacy Values ---------

legacy_values_histogram_speed_duration = [
    x * 1e-3
    for x in [
        843600.0000000102,
        142599.9999999827,
        137000.00000001042,
        169699.9999999756,
        207000.000000023,
        234499.9999999919,
        597100.0000000233,
        628999.9999999781,
        409199.9999999931,
        509999.9999999912,
        528700.0000000147,
        501800.00000001205,
        542700.0000000047,
        805799.9999999916,
        854999.9999999951,
        1398100.000000004,
        1676399.9999999749,
        6462700.000000036,
        436499.9999999865,
    ]
]

#### TAmb Histogram

In [0]:
interesting_files = [
    "AA080518_Continuous_20250427_052054_20250427_100541.MF4",
]
result_histogram_temp_duration = (
    hist_df.join(hist_meta_df, on="visual_id", how="inner")
    .where(F.col("name") == F.lit("t_amb_air_hist_p1"))
    .join(
        F.broadcast(
            measurement_dim_df.where(F.col("file_name").isin(interesting_files))
        ),
        on="container_id",
        how="inner",
    )
    .groupBy(
        F.col("name"),
        F.col("bin_ID"),
        F.col("lower_bound"),
        F.col("upper_bound"),
        F.col("bin_name"),
    )
    .agg(F.sum(F.col("hist_value")).alias("hist_value"))
    .orderBy(F.col("bin_id").asc())
    .collect()
)

actual_values_histogram_temp_duration = [
    row["hist_value"] for row in result_histogram_temp_duration
]

# --------- Legacy Values ---------

legacy_values_histogram_temp_duration = [
    x * 1e-3
    for x in [
        72000.0,
        750000.0000000001,
        3654000.0,
        4045000.000000002,
        4380100.0,
        4185299.999999998,
    ]
]

#### Histogram Custom Weights

In [0]:
interesting_files_for_custom_weights = [
    "AA080518_Continuous_20250427_052054_20250427_100541.MF4"
]
from pytest import approx

custom_weights_df = (
    hist_df.join(hist_meta_df, on="visual_id", how="inner")
    .where(F.col("name") == F.lit("hist_custom_weights"))
    .join(
        F.broadcast(
            measurement_dim_df.where(
                F.col("file_name").isin(interesting_files_for_custom_weights)
            )
        ),
        on="container_id",
        how="inner",
    )
    .groupBy(
        F.col("name"),
        F.col("bin_ID"),
        F.col("lower_bound"),
        F.col("upper_bound"),
        F.col("bin_name"),
    )
    .agg(F.sum(F.col("hist_value")).alias("hist_value"))
    .orderBy(F.col("bin_id").asc())
    .select("hist_value")
    .collect()
)
acutal_values_for_custom_weights = [row["hist_value"] for row in custom_weights_df]

# --------- Legacy Values ---------

legacy_values_for_custom_weights = [
    x * 1e-3 for x in [3879699.9999999795, 13207700.00000002]
]

#### Distance Histogram

In [0]:
interesting_files_distance = ["AA080518_Continuous_20250427_052054_20250427_100541.MF4"]
from pytest import approx

hist_distance_df = (
    hist_df.join(hist_meta_df, on="visual_id", how="inner")
    .where(F.col("name") == F.lit("histogram_1d_distance"))
    .join(
        F.broadcast(
            measurement_dim_df.where(
                F.col("file_name").isin(interesting_files_distance)
            )
        ),
        on="container_id",
        how="inner",
    )
    .groupBy(
        F.col("name"),
        F.col("bin_ID"),
        F.col("lower_bound"),
        F.col("upper_bound"),
        F.col("bin_name"),
    )
    .agg(F.sum(F.col("hist_value")).alias("hist_value"))
    .orderBy(F.col("bin_id").asc())
    .select("*")
    .collect()
)
acutal_distance_histogram_values = [row["hist_value"] for row in hist_distance_df]

# --------- Legacy Values ---------

legacy_distance_histogram_values = [
    114148.875,
    1.0,
    3.25,
    11.875,
    11.875,
    15.75,
    24.25,
    47.0,
    196.25,
    11.5,
]

#### RPM Torque Heatmap

In [0]:
interesting_files = ["AA080518_Continuous_20250427_052054_20250427_100541.MF4"]

hist2d_values = (
    hist2d_df.join(hist2d_meta_df, on="visual_id", how="inner")
    .where(F.col("name") == F.lit("rpm_torque_heatmap"))
    .join(
        F.broadcast(
            measurement_dim_df.where(F.col("file_name").isin(interesting_files))
        ),
        on="container_id",
        how="inner",
    )
    .groupBy(
        F.col("name"),
        F.col("x_bin_id"),
        F.col("y_bin_id"),
        F.col("x_lower_bound"),
        F.col("x_upper_bound"),
        F.col("y_lower_bound"),
        F.col("y_upper_bound"),
        F.col("x_bin_name"),
        F.col("y_bin_name"),
    )
    .agg(
        F.sum(F.col("hist_value")).alias("hist_value")
    )  # added because time is in second and legacy is in hour
    .withColumn("hist_value_hour", F.col("hist_value") / 3600)
    .withColumn("hist_value_rounded", F.round(F.col("hist_value_hour"), 2))
    .orderBy(F.col("y_lower_bound").asc(), F.col("x_lower_bound").asc())
    .collect()
)

actual_rpm_torque_heatmap_values = [row["hist_value"] for row in hist2d_values]

# --------- Legacy Values ---------

legacy_rpm_torque_heatmap_values = [
    x * 1e-3
    for x in [
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        52400.00000000186,
        57700.0,
        10200.000000001863,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        177199.99999998556,
        603000.0000000168,
        74600.0000000014,
        6500.000000001863,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        129899.99999999802,
        211299.99999996036,
        172200.00000000512,
        31700.000000003143,
        6999.999999996158,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        67200.0000000014,
        150499.99999999255,
        70000.00000000931,
        130900.00000001444,
        13899.99999999255,
        6399.999999996275,
        0.0,
        0.0,
        0.0,
        0.0,
        10800.000000014901,
        85799.99999999674,
        588900.0000000085,
        89500.00000000001,
        23199.99999998574,
        50899.99999998277,
        26900.000000003318,
        11800.000000000116,
        0.0,
        0.0,
        0.0,
        0.0,
        26999.999999989697,
        24300.000000003776,
        171299.99999999252,
        81900.00000001675,
        52000.00000001816,
        65600.00000000995,
        4699.999999999884,
        4300.000000001863,
        800.0,
        200.0,
        847800.0000000028,
        55999.99999999557,
        50699.999999982836,
        126900.00000001176,
        1182600.0000000326,
        349100.0000000355,
        172799.99999999668,
        253899.99999998685,
        47799.999999995925,
        100.0,
        31300.0,
        0.0,
        87699.99999999437,
        10500.000000007043,
        25999.999999994878,
        23399.999999981956,
        323799.9999999633,
        59899.999999987354,
        39099.99999998674,
        37800.0,
        3399.999999999127,
        0.0,
        0.0,
        0.0,
        13800.000000009553,
        10399.999999990054,
        12700.000000010412,
        24199.99999998085,
        281099.99999999336,
        48600.00000001991,
        20099.999999993117,
        30200.000000013097,
        2600.0000000032596,
        0.0,
        0.0,
        0.0,
        4100.000000001979,
        6600.000000002612,
        6699.999999995576,
        33100.00000001336,
        242400.0000000174,
        60400.000000002765,
        18799.999999994936,
        15200.000000001164,
        1199.9999999990687,
        0.0,
        0.0,
        0.0,
        799.9999999976135,
        4100.000000004824,
        7200.000000008848,
        23199.999999996042,
        278499.999999982,
        48199.999999988126,
        21100.000000006097,
        14499.999999999534,
        200.0,
        0.0,
        0.0,
        0.0,
        100.00000000186265,
        4699.999999997148,
        10299.999999994005,
        31400.000000012442,
        270000.00000001444,
        36900.00000000163,
        18300.000000018,
        12599.999999997257,
        100.00000000186265,
        0.0,
        0.0,
        0.0,
        0.0,
        1300.0,
        11700.000000006177,
        33900.00000000236,
        240399.99999997625,
        34600.00000000441,
        11899.99999998582,
        10600.000000006054,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        5099.999999988759,
        37400.00000001775,
        193400.0000000085,
        35699.999999991676,
        11599.999999995634,
        11600.000000002328,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        2300.000000005646,
        30599.999999972526,
        240900.00000001013,
        33500.00000000406,
        12499.999999986729,
        26299.999999983702,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        1399.9999999948777,
        31700.000000020795,
        355200.00000000873,
        33200.00000000477,
        148300.0000000278,
        105900.00000000838,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        599.9999999999418,
        29599.999999986263,
        256899.99999997945,
        120200.00000001147,
        1114400.0000000056,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        491599.9999999942,
        2414800.000000021,
        2359699.9999999655,
        100.00000000005821,
        0.0,
        0.0,
        0.0,
    ]
]

#### Custom Weights and Distance Heatmap

In [0]:
interesting_files_for_custom_weights = [
    "AA080518_Continuous_20250427_052054_20250427_100541.MF4"
]
hist2d_custom_weights_df = (
    hist2d_df.join(hist2d_meta_df, on="visual_id", how="inner")
    .where(F.col("name") == F.lit("histogram_2d_custom_weights"))
    .join(
        F.broadcast(
            measurement_dim_df.where(
                F.col("file_name").isin(interesting_files_for_custom_weights)
            )
        ),
        on="container_id",
        how="inner",
    )
    .groupBy(
        F.col("name"),
        F.col("x_bin_id"),
        F.col("y_bin_id"),
        F.col("x_lower_bound"),
        F.col("x_upper_bound"),
        F.col("y_lower_bound"),
        F.col("y_upper_bound"),
        F.col("x_bin_name"),
        F.col("y_bin_name"),
    )
    .agg(F.sum(F.col("hist_value")).alias("hist_value"))
    .orderBy(F.col("y_lower_bound").asc(), F.col("x_lower_bound").asc())
    .collect()
)

actual_custom_weights_heatmap_values = [
    row["hist_value"] for row in hist2d_custom_weights_df
]

# -------------------------------------------------------

interesting_files_for_custom_weights = [
    "AA080518_Continuous_20250427_052054_20250427_100541.MF4"
]
hist2d_distance_df = (
    hist2d_df.join(hist2d_meta_df, on="visual_id", how="inner")
    .where(F.col("name") == F.lit("histogram_2d_distance"))
    .join(
        F.broadcast(
            measurement_dim_df.where(
                F.col("file_name").isin(interesting_files_for_custom_weights)
            )
        ),
        on="container_id",
        how="inner",
    )
    .groupBy(
        F.col("name"),
        F.col("x_bin_id"),
        F.col("y_bin_id"),
        F.col("x_lower_bound"),
        F.col("x_upper_bound"),
        F.col("y_lower_bound"),
        F.col("y_upper_bound"),
        F.col("x_bin_name"),
        F.col("y_bin_name"),
    )
    .agg(F.sum(F.col("hist_value")).alias("hist_value"))
    .orderBy(F.col("y_lower_bound").asc(), F.col("x_lower_bound").asc())
    .collect()
)

actual_distance_heatmap_values = [row["hist_value"] for row in hist2d_distance_df]

# --------- Legacy Values ---------

legacy_custom_weights_heatmap_values = [
    114148.375,
    0.625,
    1.25,
    0.875,
    0.0,
    0.0,
    0.25,
    11.0,
    14.375,
    0.125,
    0.0,
    0.0,
    27.375,
    30.125,
    0.25,
    2.0,
    1.5,
    163.875,
    63.125,
    6.5,
]

## Assertions

In [0]:
# Assertion of 1d Histogram values
assert actual_values_histogram_rpm_duration == approx(
    legacy_values_histogram_rpm_duration, rel=0.01
)
assert actual_values_histogram_speed_duration == approx(
    legacy_values_histogram_speed_duration, rel=0.001
)
assert actual_values_histogram_temp_duration == approx(
    legacy_values_histogram_temp_duration, rel=0.001
)
assert acutal_values_for_custom_weights == approx(
    legacy_values_for_custom_weights, rel=0.001
)
assert acutal_distance_histogram_values == approx(
    legacy_distance_histogram_values, rel=0.001
)

# Assertion of 2d Histogram values
assert actual_rpm_torque_heatmap_values == approx(
    legacy_rpm_torque_heatmap_values, rel=0.01
)
assert actual_custom_weights_heatmap_values == approx(
    legacy_custom_weights_heatmap_values, rel=0.001
)
assert actual_distance_heatmap_values == approx(
    legacy_custom_weights_heatmap_values, rel=0.001
)

In [0]:
histogram_values = [
    [actual_values_histogram_rpm_duration, legacy_values_histogram_rpm_duration],
    [actual_values_histogram_speed_duration, legacy_values_histogram_speed_duration],
    [actual_values_histogram_temp_duration, legacy_values_histogram_temp_duration],
    [acutal_values_for_custom_weights, legacy_values_for_custom_weights],
    [acutal_distance_histogram_values, legacy_distance_histogram_values],
    [actual_rpm_torque_heatmap_values, legacy_rpm_torque_heatmap_values],
    [acutal_distance_histogram_values, legacy_distance_histogram_values],
    [actual_custom_weights_heatmap_values, legacy_custom_weights_heatmap_values],
]

names = [
    "RPM Duration",
    "Spped Duration",
    "Temp Duration",
    "Custom Weights",
    "RPM Torque",
    "2D Distance",
    "2D Custom Weights",
]
for elem, name in zip(histogram_values, names, strict=False):
    print(f"\n----------------------------  {name}  -----------------------\n")
    for i, (actual, legacy) in enumerate(zip(elem[0], elem[1], strict=False)):
        if legacy == 0:
            rel_diff = float("inf") if actual != 0 else 0
        else:
            rel_diff = abs(actual - legacy) / abs(legacy)
        # if rel_diff > 0.001:
        print(
            f"Index {i}: actual={actual}, legacy={legacy}, rel_diff={rel_diff}, diff={(actual - legacy)}"
        )